In [1]:
import pandas as pd # librería que usamos para trabajar con tablas de datos
from sklearn.preprocessing import MinMaxScaler # sirve para dejar los valores en una escala de 0 a 1
import unicodedata # ayuda a manejar las tildes de los textos

In [2]:
# 1. Revisar los datos al inicio
df = pd.read_csv("Data/datos_para_limpiar.csv") # el archivo debe estar dentro de la carpeta "Data".

In [3]:
print("Primeras 5 filas:")# head() muestra las primeras 5 filas para revisar cómo vienen los datos.
print(df.head())

Primeras 5 filas:
   ID  Nombre  Edad       Salario  Ciudad      Genero   Nivel       Fecha  \
0   1     Ana   NaN  3.955413e+03  bogota   Masculino    Bajo  2018-12-29   
1   2   Maria   NaN  3.054891e+03     NaN    Femenino    alto  2011-07-16   
2   3  Carlos   NaN  2.928345e+08  bogota    Femenino    Bajo  2022-04-07   
3   4   Maria  11.0  4.718402e+03     NaN         NaN  medio   2022-10-11   
4   5   Laura  94.0  7.477876e+07    cali  masculino     alto  2017-07-11   

      Ingresos       Gastos    Codigo                 Notas  
0  4030.026008  1213.936751  ABC12345  Pendiente evaluación  
1  7509.810584  6038.896355       NaN         Buen empleado  
2  9106.362328  6705.724812       NaN                   NaN  
3          NaN          NaN  XYZ98765                   NaN  
4  3102.167244  7570.107277       NaN  Pendiente evaluación  


In [4]:
print("Información general:") # info() muestra el tipo de dato de cada columna y cuántos datos tienen valor.
# por ejemplo, "fecha" aparece como texto y luego la vamos a convertir a fecha.
df.info()

Información general:
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 12 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   ID        100 non-null    int64  
 1   Nombre    100 non-null    str    
 2   Edad      50 non-null     float64
 3   Salario   65 non-null     float64
 4   Ciudad    82 non-null     str    
 5   Genero    73 non-null     str    
 6   Nivel     73 non-null     str    
 7   Fecha     100 non-null    str    
 8   Ingresos  48 non-null     float64
 9   Gastos    53 non-null     float64
 10  Codigo    69 non-null     str    
 11  Notas     67 non-null     str    
dtypes: float64(4), int64(1), str(7)
memory usage: 9.5 KB


In [5]:
print("Nulos por columna:") # isnull() identifica los espacios vacíos y sum() cuenta cuántos hay en cada columna.
print(df.isnull().sum())

Nulos por columna:
ID           0
Nombre       0
Edad        50
Salario     35
Ciudad      18
Genero      27
Nivel       27
Fecha        0
Ingresos    52
Gastos      47
Codigo      31
Notas       33
dtype: int64


In [6]:
# 2. Quitar las columnas que no necesitamos (ID y Notas)
df = df.drop(columns=["ID", "Notas"]) # con columns indicamos que vamos a quitar columnas y no filas.

In [7]:
# 3. Dejar los nombres de las columnas en un formato uniforme
# los ponemos en minúsculas, sin espacios y sin tildes
# 1. normalize("NFKD") separa las letras de sus tildes.
# 2. combining(c) ayuda a reconocer los caracteres que son tildes.
# 3. dejamos solo los caracteres que no son tildes.
def quitar_tildes(texto):
    return "".join(
        c for c in unicodedata.normalize("NFKD", texto)
        if not unicodedata.combining(c)
    )
 
df.columns = [
    quitar_tildes(c.strip().lower()).replace(" ", "_") for c in df.columns
]
 

In [8]:
# 4. Corregir cómo están escritos los nombres de las ciudades
#    quitamos espacios, mayúsculas y tildes para poder comparar los nombres,
#    y después dejamos el nombre correcto de cada ciudad.
mapa_ciudades = {"bogota": "Bogotá", "medellin": "Medellín", "cali": "Cali"}
 
df["ciudad"] = (
    df["ciudad"]
    .str.strip()
    .str.lower()
    .apply(lambda x: quitar_tildes(x) if isinstance(x, str) else x)
    .map(mapa_ciudades)          # si hay valores vacíos, se mantienen vacíos
)

In [9]:
# 5. Completar los valores que están vacíos
# edad: usamos el promedio de las edades válidas, entre 0 y 120
# hay edades que no tienen sentido, como -3, 125 o 129. Si las usamos,
# el promedio puede quedar mal. Primero tomamos las edades entre 0 y 120
# y calculamos el promedio usando solo esos valores.
edades_validas = df.loc[df["edad"].between(0, 120), "edad"]
df["edad"] = df["edad"].fillna(edades_validas.mean())

In [10]:
# salario: completamos los vacíos con la mediana
# la mediana ayuda cuando hay valores demasiado altos o bajos, y en los salarios
# aparecen algunos valores de cientos de millones.
df["salario"] = df["salario"].fillna(df["salario"].median())

In [11]:
# ciudad y género: usamos la moda
# la moda es el valor que más se repite. Sirve para datos de texto, porque no
# podemos sacarles un promedio. mode() devuelve los valores más repetidos y [0]
# toma el primero de la lista.
df["ciudad"] = df["ciudad"].fillna(df["ciudad"].mode()[0])
df["genero"] = df["genero"].fillna(df["genero"].mode()[0])

In [12]:
# 6. Asegurar que los datos tengan el tipo correcto
# to_numeric convierte los datos a números. Con errors="coerce", si algo no
# se puede convertir, no se detiene el código: queda como nulo (NaN).
df["edad"] = pd.to_numeric(df["edad"], errors="coerce")
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")

In [13]:
# 7. Quitar los datos que están fuera del rango permitido
# filtrar en pandas es dejar solo las filas que cumplen una condición.
# dejamos edades entre 0 y 120 y quitamos casos como -3, 125 o 129.
df = df[df["edad"].between(0, 120)]          # solo edades entre 0 y 120
df = df[df["salario"] <= 1_000_000_000]      # solo salarios de hasta 1.000 millones

In [14]:
# 8. Pasar el nivel a números
# map() cambia cada nivel por el número que aparece en el diccionario. Si el dato
# no está en el diccionario, como un vacío o "n/a", queda como NaN, que es lo
# que pide el taller para conservar esos datos como nulos.
df["nivel"] = df["nivel"].map({"bajo": 1, "medio": 2, "alto": 3})

In [15]:
# 9. Convertir las categorías a columnas con valores 0 y 1 (one-hot)
# drop_first=True quita una categoría de cada variable para evitar información
# repetida: si no es Cali ni Medellín, entonces es Bogotá, así que esa columna
# no hace falta.
# dtype=int hace que las nuevas columnas tengan 0 y 1 en vez de True y False.
# las columnas originales "genero" y "ciudad" se cambian por las nuevas.
# después de este paso, la columna original "ciudad" ya no está.
df = pd.get_dummies(df, columns=["genero", "ciudad"], drop_first=True, dtype=int)

In [16]:
# 10. Poner edad e ingresos en una escala de 0 a 1
# MinMaxScaler no usa los NaN para calcular el mínimo y el máximo.
# la edad y los ingresos tienen rangos muy diferentes. Algunos algoritmos pueden
# funcionar peor con esa diferencia, así que MinMaxScaler lleva ambos valores de 0 a 1:
#   valor nuevo = (valor - mínimo) / (máximo - mínimo)
scaler = MinMaxScaler()
df[["edad", "ingresos"]] = scaler.fit_transform(df[["edad", "ingresos"]])

In [17]:
# 11. Quitar las filas repetidas
# drop_duplicates() elimina las filas que tienen todos sus datos iguales,
# y conserva la primera que aparece.
df = df.drop_duplicates()


In [18]:
# 12. Crear una nueva columna llamada utilidad (ingresos menos gastos)
# en el paso 10, "ingresos" quedó entre 0 y 1, pero
# "gastos" sigue con su valor real. Por eso la resta mezcla dos escalas. Se deja
# en este orden porque así lo pide el taller. Si necesitamos la utilidad en pesos
# reales, habría que hacer este cálculo antes del paso 10.
df["utilidad"] = df["ingresos"] - df["gastos"]

In [19]:
# 13. Guardar el archivo con los datos limpios
# index=False evita que pandas agregue una columna extra con el índice de las filas.
df.to_csv("datos_limpios.csv", index=False)
